In [1]:
#load apikeys from .env
from dotenv import load_dotenv
import os
import getpass

load_dotenv()


if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Introduce tu  AWS Key: ")

In [2]:
#model agnostic function for llm inference with init_chat_model and invoke()

from langchain.chat_models import init_chat_model

def generate_example_orders(model_name, prompt, provider=None, token_limit=200, temperature = 0.7):
    model = init_chat_model(
        model_name,
        model_provider=provider,
        max_tokens=token_limit,
        temperature = temperature
    )
    response = model.invoke(prompt)
    return response

In [3]:
model_name = "claude-haiku-4-5-20251001"
provider = "anthropic"
prompt = "Generate the transcription of a pizza order"


response = generate_example_orders(
    model_name,
    prompt,
    provider
)

response

AIMessage(content="# Pizza Order Transcription\n\n**Customer:** Hi, I'd like to place a delivery order, please.\n\n**Employee:** Sure thing! What can I get started for you today?\n\n**Customer:** Um, I'd like a large pepperoni pizza and a medium mushroom and onion.\n\n**Employee:** Okay, so one large pepperoni and one medium with mushroom and onion. Anything else?\n\n**Customer:** Yeah, can I get a two-liter of Coke and an order of garlic breadsticks?\n\n**Employee:** Perfect. One two-liter Coke and garlic breadsticks. What size breadsticks—regular or family?\n\n**Customer:** Regular is fine.\n\n**Employee:** Got it. Will that be cash or card?\n\n**Customer:** Card, please.\n\n**Employee:** Alright, your total comes to $34.99. What's the address", additional_kwargs={}, response_metadata={'id': 'msg_011CfF8HQDBrzs8r5UBXbaDL', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'max_tokens', 'stop_sequence': None, 'usage': {'cache_creation': {'ep

In [4]:
#Components of the AI Message

#response.content
#response.response_metadata
#response.tool_calls
#response.invalid_tool_calls
response.usage_metadata

{'input_tokens': 15,
 'output_tokens': 200,
 'total_tokens': 215,
 'input_token_details': {'cache_read': 0,
  'cache_creation': 0,
  'ephemeral_5m_input_tokens': 0,
  'ephemeral_1h_input_tokens': 0}}

In [5]:
#print out all messages

import json

def print_aimessage(ai_message):
    for key, value in ai_message.model_dump().items():                #convert Ai-Message to Python dictionary and get items
        if isinstance(value, (dict, list)):                         #check if the value is a nested structure
            print(f"\n[{key}]:")                                    
            print(json.dumps(value, indent=2, ensure_ascii=False))  #converts a Python object into a JSON-formatted string. It’s commonly used to serialize data for storage or sending
        else:
            print(f"\n[{key}]: {value}")

print_aimessage(response)


[content]: # Pizza Order Transcription

**Customer:** Hi, I'd like to place a delivery order, please.

**Employee:** Sure thing! What can I get started for you today?

**Customer:** Um, I'd like a large pepperoni pizza and a medium mushroom and onion.

**Employee:** Okay, so one large pepperoni and one medium with mushroom and onion. Anything else?

**Customer:** Yeah, can I get a two-liter of Coke and an order of garlic breadsticks?

**Employee:** Perfect. One two-liter Coke and garlic breadsticks. What size breadsticks—regular or family?

**Customer:** Regular is fine.

**Employee:** Got it. Will that be cash or card?

**Customer:** Card, please.

**Employee:** Alright, your total comes to $34.99. What's the address

[additional_kwargs]:
{}

[response_metadata]:
{
  "id": "msg_011CfF8HQDBrzs8r5UBXbaDL",
  "container": null,
  "model": "claude-haiku-4-5-20251001",
  "stop_details": null,
  "stop_reason": "max_tokens",
  "stop_sequence": null,
  "usage": {
    "cache_creation": {
    

In [12]:
#start the ollama server
#uv add langchain_ollama
# utliza un modelo descargado

model_name = "qwen3:8b"
provider = "ollama"
prompt = "Generate the transcription of a pizza order"

response = generate_example_orders(
    model_name,
    prompt,
    provider
)

print_aimessage(response)


[content]: **Pizza Order Transcription**  

**Customer:** "Hey, I ordered a pizza, right?"  
**Delivery Person:** "Yes, sir! Your large pepperoni, mushrooms, and olives pizza is on its way. Should be there in about 25 minutes."  

**Customer:** "Great, thanks. Oh, wait—did you get the extra cheese?"  
**Delivery Person:** "Nope, I didn’t. You wanted regular cheese, right?"  

**Customer:** "Right. And make sure to add the jalapeños. I mentioned them last time."  
**Delivery Person:** "Got it. Jalapeños added. Anything else?"  

**Customer:** "No, that’s it. Just bring it to my place—123 Maple Street."  
**Delivery Person:** "On it! Keep an eye out for the pizza truck. And if you need anything else, just call!"  

**Customer:** "Will do. Thanks again!"  
**Delivery Person:** "No problem! See you soon!"  

*(Delivery person drives away with the pizza box.)*  

---  
*Note: This transcription mimics a real-world interaction, including common order details, clarifications, and casual tone

In [18]:
import time
import pandas as pd
from pathlib import Path
from langchain.chat_models import init_chat_model

# 1. Define model configurations
models_config = [
    # Ollama Models
    {"model": "gemma3:4b", "provider": "ollama"},
    {"model": "gemma4:e4b", "provider": "ollama"},

    {"model": "llama3.1:8b", "provider": "ollama"},

    {"model": "ministral-3:3b", "provider": "ollama"},
    {"model": "mistral:7b", "provider": "ollama"},
    {"model": "mistral-nemo:12b", "provider": "ollama"},

    {"model": "qwen3.5:0.8b", "provider": "ollama"},
    {"model": "qwen3:4b-instruct", "provider": "ollama"},
    {"model": "qwen3:8b", "provider": "ollama"},
    {"model": "qwen3.5:9b", "provider": "ollama"},
    
    # Anthropic Model
    {"model": "claude-haiku-4-5-20251001", "provider": "anthropic"},
]

prompt = "Generate the transcription of a pizza order."
results = []



# 2. Invoke models and record metadata
for config in models_config:
    model_name = config["model"]
    provider = config["provider"]
    temperature = 1.0
    
    try:
        # Initialize LLM via LangChain init_chat_model
        llm = init_chat_model(model_name, model_provider=provider, temperature=temperature)
        
        # Track wall-clock duration
        start_time = time.perf_counter()
        response = llm.invoke(prompt)
        end_time = time.perf_counter()
        
        duration = round(end_time - start_time, 3)
        answer = response.content
        
        # Extract usage metadata
        usage_metadata = getattr(response, "usage_metadata", {}) or {}
        input_tokens = usage_metadata.get("input_tokens", None)
        output_tokens = usage_metadata.get("output_tokens", None)
        
        
        results.append({
            "modelname": model_name,
            "provider": provider,
            "duration_sec": duration,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "answer": answer
        })
        print(f"Executed {model_name} in {duration} seconds. Input tokens: {input_tokens}, Output tokens: {output_tokens}, Answer: {answer[:80]}...")  # Print first 50 chars of answer for brevity
        
    except Exception as e:
        print(f"Error executing {model_name}: {e}")

# 3. Create Pandas DataFrame
df = pd.DataFrame(results)

df

Executed gemma3:4b in 30.468 seconds. Input tokens: 18, Output tokens: 645, Answer: Okay, here’s a transcription of a pizza order, aiming for a realistic conversati...
Error executing gemma4:e4b: model 'gemma4:e4b' not found (status code: 404)
Executed llama3.1:8b in 11.517 seconds. Input tokens: 18, Output tokens: 359, Answer: Here is a transcription of a pizza order:

**Phone rings**

**Pizza Place**: Tha...
Error executing ministral-3:3b: model 'ministral-3:3b' not found (status code: 404)
Executed mistral:7b in 14.782 seconds. Input tokens: 13, Output tokens: 646, Answer:  Customer: Hi, I'd like to place an order please.

Assistant: Of course! May I h...
Error executing mistral-nemo:12b: llama-server process has terminated: exit status 0xc0000409: The system detected an overrun of a stack-based buffer in this application. This overrun could potentially allow a malicious user to gain control of this application.: CUDA error: shared object initialization failed (status code: 500)
Err

,modelname,provider,duration_sec,input_tokens,output_tokens,answer
0,gemma3:4b,ollama,30.468,18,645,"Okay, here’s a transcription of a pizza order,..."
1,llama3.1:8b,ollama,11.517,18,359,Here is a transcription of a pizza order:\n\n*...
2,mistral:7b,ollama,14.782,13,646,"Customer: Hi, I'd like to place an order plea..."
3,qwen3:8b,ollama,21.737,18,883,**Pizza Order Transcription** \n\n**Customer:...
4,claude-haiku-4-5-20251001,anthropic,3.650,16,306,# Pizza Order Transcription\n\n**Customer:** H...


In [19]:
print(type(df))

<class 'pandas.DataFrame'>


In [20]:
# save df to csv
from datetime import datetime
from pathlib import Path

script_dir = Path.cwd()
data_path = script_dir.parent / "data" / "performance"

data_path.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = data_path / f"performance_test_{timestamp}.csv"
df.to_csv(data_path / output_file, index=False)

In [21]:
df.sort_values(by="duration_sec", ascending=True, inplace=False)

,modelname,provider,duration_sec,input_tokens,output_tokens,answer
4,claude-haiku-4-5-20251001,anthropic,3.650,16,306,# Pizza Order Transcription\n\n**Customer:** H...
1,llama3.1:8b,ollama,11.517,18,359,Here is a transcription of a pizza order:\n\n*...
2,mistral:7b,ollama,14.782,13,646,"Customer: Hi, I'd like to place an order plea..."
3,qwen3:8b,ollama,21.737,18,883,**Pizza Order Transcription** \n\n**Customer:...
0,gemma3:4b,ollama,30.468,18,645,"Okay, here’s a transcription of a pizza order,..."


In [22]:
answer_list = df ["answer"].to_list()
for idx, answer in enumerate(answer_list):
    print(f"Answer from model {df['modelname'][idx]} Output tokens: {df['output_tokens'][idx]}:\n {answer}\n")
    print("------------------------------------------------------------\n")

Answer from model gemma3:4b Output tokens: 645:
 Okay, here’s a transcription of a pizza order, aiming for a realistic conversation:

**Setting:** A phone call to "Tony's Pizza"

**Characters:**

* **Sarah (S):** The Customer
* **Tony (T):** The Pizza Order Taker

---

**(Sound of a phone ringing, then being answered)**

**T:** Tony's Pizza, this is Tony speaking, how can I help you?

**S:** Hi Tony, I’d like to place an order for pickup.

**T:** Great! Can I get your name and address, please?

**S:** It’s Sarah Miller, and my address is 14 Maple Street, Apartment 2B.

**T:** Okay, Sarah Miller, 14 Maple Street, Apartment 2B.  And what are you thinking of ordering tonight?

**S:** I’m craving a pizza. I’d like a large pepperoni pizza, please.

**T:** Large pepperoni. Got it. Anything else on that?

**S:** Hmm, could I also add some mushrooms?

**T:** Sure thing.  Large pepperoni with mushrooms.  Would you like any sauce on that? We have red or white.

**S:** Red, please. And could you 

In [24]:
#to manage results more easely we only want the turn with the order.
#adapt prompt: Only generate the customers turn with the order.
model_name = "qwen3:8b"
provider = "ollama"
prompt = """
Generate a pizza order transcribed as if the customer was placing them verbally over the phone. 
Only generate the customers turn with the order.
"""


response = generate_example_orders(
    model_name,
    prompt,
    provider,
    temperature
)

print(response.content)
print(len(response.content))


"Hi, I’d like to order two large pizzas. One with pepperoni, extra cheese, and mushrooms, and the other is a veggie pizza with pineapple, bell peppers, and olives. Oh, and make sure the crust is crispy on both. Wait, do you have gluten-free crust available? Yeah, the veggie one can be gluten-free. Also, add a side of garlic knots and a drink—maybe a Coke. Delivery, please, and have it ready by 7:30 PM. Thanks!"
414


In [ ]:
# The standard way to ensure consistent output format from LLMs is to use Pydantic models to define and validate the expected structure, 
# which can then be enforced via the API's structured-output feature

from typing import List
from pydantic import BaseModel, Field

class Pizza(BaseModel):
    """Transcription of the pizza order."""

    pizza_order: str = Field(
        ..., description="The pizza order transcription."
    )

class Pizza_order_list(BaseModel):
    """ Generated Pizza orders"""
    # Creates a model so that we can extract multiple entities.
    order_list: List[Pizza]



In [ ]:
from langchain.chat_models import init_chat_model

def generate_structured_example_orders(model_name, prompt, provider=None, temperature = 0.7):
    model = init_chat_model(
        model_name,
        model_provider=provider,
        temperature = temperature
    )
    structured_llm = model.with_structured_output(schema=Pizza_order_list, include_raw=True)
    response = structured_llm.invoke(prompt)
    return response

In [ ]:
model_name = "qwen3:4b-instruct"
provider = "ollama"
prompt = """
Generate 5 pizza orders transcribed as if the customer was placing them verbally over the phone. 
Only generate the customers turn with the order.
"""

response = generate_structured_example_orders(
    model_name,
    prompt,
    provider,
    temperature
)
response

In [ ]:
#transform pydantic object to Python dict using model_dump()
print(response["raw"].usage_metadata)
print(response["parsed"])
response_dict = response["parsed"].model_dump()
print(type(response_dict))
response_dict

In [ ]:
#get the values of the dict 
for order in response_dict['order_list']:
    print(order ['pizza_order'])

In [ ]:
# save  response (pydantic object) as json 
from datetime import datetime
from pathlib import Path


def save_orders_to_file(content) -> Path:
    data_path = Path.cwd().parent / "data" / "example_orders"
    data_path.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = data_path / f"pedidos_pizza_{timestamp}.json"

    output_file.write_text(
        content.model_dump_json(indent=2),
        encoding="utf-8",
    )

    return output_file

In [ ]:
save_orders_to_file(response)

In [ ]:
from pathlib import Path


def load_orders_as_list(filename: str) -> list[str]:
    """Lädt eine JSON-Datei aus data/example_orders und gibt die Orders als Liste zurück."""
    data_path = Path.cwd().parent / "data" / "example_orders"
    path = data_path / filename
    data = json.loads(path.read_text(encoding="utf-8"))

    return [order["pizza_order"] for order in data["order_list"]]

    


In [ ]:
python_list = load_orders_as_list("pedidos_pizza_20260914_115625.json")
print(type(python_list))
python_list
